import modules

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/Users/hongmo/Desktop/Gradient Boosting"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_and_split_data
from src.preprocessing import build_preprocessor
from src.models import build_xgboost_pipeline

from sklearn.metrics import roc_auc_score
import time

load and split data

In [3]:
data_path = PROJECT_ROOT / "data" / "application_train.csv"

X_train, X_val, X_test, y_train, y_val, y_test = (
    load_and_split_data(data_path)
)

build XGBoost Pipeline

In [4]:
preprocessor = build_preprocessor(
    X_train,
    scale_numeric=False
)

xgb_pipeline = build_xgboost_pipeline(preprocessor)

train XGBoost

In [5]:
start_time = time.perf_counter()
xgb_pipeline.fit(X_train, y_train)
training_time = time.perf_counter() - start_time

print(f"Training Time: {training_time:.2f} seconds")

Training Time: 6.51 seconds


Calculate Training and Validation AUC

In [6]:
# Predict probabilities
y_train_prob = xgb_pipeline.predict_proba(X_train)[:, 1]
y_val_prob = xgb_pipeline.predict_proba(X_val)[:, 1]

# Calculate ROC-AUC
train_auc = roc_auc_score(y_train, y_train_prob)
val_auc = roc_auc_score(y_val, y_val_prob)

print(f"Training AUC:   {train_auc:.4f}")
print(f"Validation AUC: {val_auc:.4f}")
print(f"AUC Gap:        {train_auc - val_auc:.4f}")

Training AUC:   0.7829
Validation AUC: 0.7573
AUC Gap:        0.0255


In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    xgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

print("Fold AUCs:", cv_scores)
print(f"Mean CV AUC: {cv_scores.mean():.4f}")
print(f"CV Std: {cv_scores.std():.4f}")

Fold AUCs: [0.75203005 0.75593981 0.75530252 0.75598216 0.75065392]
Mean CV AUC: 0.7540
CV Std: 0.0022


In [8]:
initial_cv_mean = cv_scores.mean()
initial_cv_std = cv_scores.std()

print(f"Initial CV AUC: {initial_cv_mean:.4f}")
print(f"Initial CV Std: {initial_cv_std:.4f}")

Initial CV AUC: 0.7540
Initial CV Std: 0.0022
